# 3.1 预测任务定义、固定划分与验证基线 · 复核

这一步当时用 `src/prepare.py` 执行，产物是 `outputs/split_assignments.csv`（每条原始行属于哪一份）、`split_summary.json` 和 `baseline_validation.json`。
这个 notebook 从划分表和原始 CSV 重算各份的行数、跨划分检查和基线，和当时的产物对账，不改任何历史文件。


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
WINES = {"red": "红酒", "white": "白酒"}
FEATURES = ["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides",
            "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"]
STEP = ROOT / "steps/03_数据划分/3.1_预测任务定义_固定划分与验证基线"
summary = json.loads((STEP / "outputs/split_summary.json").read_text(encoding="utf-8"))
baseline = json.loads((STEP / "outputs/baseline_validation.json").read_text(encoding="utf-8"))
run = dsflow.start_run("3.1", project=ROOT, hypothesis="从划分表和原始 CSV 重算三份的行数、跨划分检查和训练集中位数基线，与 3.1 当时的产物一致")
raw = {}
for w, name in WINES.items():
    path = ROOT / f"data/winequality-{w}.csv"
    run.log_input(path, name=name)
    raw[w] = pd.read_csv(path, sep=";").assign(source_row=lambda d: np.arange(1, len(d) + 1))
run.log_input(STEP / "outputs/split_assignments.csv", name="划分表")
splits = pd.read_csv(STEP / "outputs/split_assignments.csv")
print(f"划分表 {len(splits):,} 行 = 红酒 {len(raw['red']):,} + 白酒 {len(raw['white']):,}：{len(splits) == len(raw['red']) + len(raw['white'])}")


划分表 6,497 行 = 红酒 1,599 + 白酒 4,898：True


In [2]:
rows = []
for w, name in WINES.items():
    df = raw[w].merge(splits[splits["wine"] == w].drop(columns="wine"), on="source_row", how="left", validate="one_to_one")
    assert df["split"].notna().all()
    crossed = int((df.groupby("feature_group_id")["split"].nunique() > 1).sum())
    mixed = int((df.groupby("feature_group_id")["quality"].nunique() > 1).sum())
    counts = df["split"].value_counts()
    groups = df.groupby("split")["feature_group_id"].nunique()
    for s in ("training", "validation", "final_evaluation"):
        assert int(counts[s]) == summary["files"][w]["totals"]["rows"][s] and int(groups[s]) == summary["files"][w]["totals"]["feature_groups"][s]
    rows.append({"酒类": name, "训练行": int(counts["training"]), "验证行": int(counts["validation"]), "最终评估行": int(counts["final_evaluation"]),
                 "特征组合": df["feature_group_id"].nunique(), "跨划分的组合": crossed, "quality 不一致的组合": mixed})
    raw[w] = df
print(pd.DataFrame(rows).to_string(index=False))
print("三份的行数、组合数与 3.1 当时的 split_summary.json 一致；没有特征组合跨划分。")


酒类  训练行  验证行  最终评估行  特征组合  跨划分的组合  quality 不一致的组合
红酒  950  331    318  1359       0               0
白酒 2921  983    994  3961       0               0
三份的行数、组合数与 3.1 当时的 split_summary.json 一致；没有特征组合跨划分。


In [3]:
rows = []
for w, name in WINES.items():
    df = raw[w]
    train, valid = df[df["split"] == "training"], df[df["split"] == "validation"]
    median = float(np.median(train["quality"]))
    err = np.abs(valid["quality"].to_numpy(float) - median)
    mae, rmse = float(err.mean()), float(np.sqrt((err ** 2).mean()))
    ref = baseline["files"][w]
    assert median == ref["training_quality_median_prediction"] and abs(mae - ref["validation_mae"]) < 1e-10 and abs(rmse - ref["validation_rmse"]) < 1e-10
    rows.append({"酒类": name, "训练行": len(train), "验证行": len(valid), "训练集中位数": median, "验证 MAE": round(mae, 4), "验证 RMSE": round(rmse, 4)})
    run.log_metrics({f"{name}_基线_MAE_验证": mae, f"{name}_基线_RMSE_验证": rmse})
print(pd.DataFrame(rows).to_string(index=False))
first = raw["red"][raw["red"]["source_row"] == 1].iloc[0]
print(f"红酒第 1 行：quality={int(first['quality'])}，划分={first['split']}，基线猜 6.0，绝对差 {abs(first['quality'] - 6.0)}")
run.set_conclusion("三份行数、组合数、跨划分检查和基线 MAE / RMSE 与 3.1 当时的产物一致", validity="有效")
run.end()


酒类  训练行  验证行  训练集中位数  验证 MAE  验证 RMSE
红酒  950  331     6.0  0.6435   0.8777
白酒 2921  983     6.0  0.6368   0.9033
红酒第 1 行：quality=5，划分=validation，基线猜 6.0，绝对差 1.0
